# Stage 2 — A 73-account social accounting matrix for 2023

The matrix combines 64 industry accounts with labour, capital, three tax accounts, a representative private household, government, saving–investment and the rest of the world. Rows receive and columns pay; values are in thousand TRY.

## 1. Paths and input verification

Public inputs are checked against their manifest. Regenerated industry IO tables are checked through accounting identities.

In [ ]:
from pathlib import Path
import json, hashlib, tempfile, subprocess, shutil, os
import numpy as np
import pandas as pd
from IPython.display import display

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/'datas/TurkStat_institutional_accounts_2023_release_2025.xlsx').exists() and (p/'codes').is_dir(): return p
    raise FileNotFoundError('Open the notebook from the stage-2 codes directory.')
ROOT=find_root(); DATA=ROOT/'datas'; OUT=ROOT/'outputs'
OUT.mkdir(exist_ok=True)
manifest=json.loads((DATA/'input_manifest.json').read_text(encoding='utf-8'))
for entry in manifest:
    assert (ROOT/entry['file']).is_file(),entry['file']
    if entry.get('sha256'): assert hashlib.sha256((ROOT/entry['file']).read_bytes()).hexdigest()==entry['sha256'],entry['file']
IO=ROOT.parent/'1_input_output/analysis/outputs/I_industry_io_tables_2023.xlsx'
TAX=DATA/'TurkStat_institutional_accounts_2023_release_2025.xlsx'
codes=[f'S{i:02d}' for i in range(64)]
checks=[]
def verify(name,actual,expected,atol=1e-3,rtol=0):
    a=np.asarray(actual,dtype=float);b=np.asarray(expected,dtype=float)
    error=float(np.max(np.abs(a-b)))
    ok=bool(np.allclose(a,b,atol=atol,rtol=rtol))
    checks.append({'Check':name,'Maximum absolute difference':error,'Tolerance':atol,'Passed':ok})
    assert ok,f'{name}: {error}'
print('2023 | 64 sectors | thousand TRY | inputs verified')

## 2. Direct taxes from institutional sector accounts

General-government resources under transaction D.5 provide the aggregate direct-tax input. D.51 and D.59 reconcile to D.5. Other institutional uses are retained for provenance.

In [ ]:
tax_raw=pd.read_excel(TAX,sheet_name='t_2023',header=None)
assert '2023' in str(tax_raw.iat[0,0])
assert 'Thousand TRY' in str(tax_raw.iat[1,14])
assert 'Resources' in str(tax_raw.iat[2,9])
assert 'S.13' in str(tax_raw.iat[3,11])
def tax_row(code):
    indices=tax_raw.index[tax_raw[7].astype(str).str.strip().eq(code)]
    assert len(indices)==1,(code,indices.tolist())
    return int(indices[0])
r5,r51,r59=[tax_row(c) for c in ['D.5','D.51','D.59']]
TD=float(tax_raw.iat[r5,11])
assert TD>0
verify('D.5 = D.51 + D.59',TD,float(tax_raw.iat[r51,11])+float(tax_raw.iat[r59,11]),atol=1e-6)
tax_extract=[]
for code,r in [('D.5',r5),('D.51',r51),('D.59',r59)]:
    for col,sector,side in [(11,'S.13','Source'),(3,'S.14+S.15','Uses'),(4,'S.13','Uses'),(5,'S.12','Uses'),(6,'S.11','Uses')]:
        tax_extract.append({'year':2023,'transaction':code,'sector':sector,'side':side,'value_thousand_TRY':float(tax_raw.iat[r,col]),'sheet':'t_2023','cell':f'{chr(65+col)}{r+1}','sam_input':code=='D.5' and col==11})
tax_extract=pd.DataFrame(tax_extract)
tax_extract.to_csv(DATA/'direct_taxes_2023.csv',index=False,encoding='utf-8-sig')
display(tax_extract)

## 3. Separate and reconcile IO flows

The industry IO table supplies intermediate and final uses, factor compensation, net taxes and imports. Aggregate flows are reconciled against the 2023 institutional sector accounts.

In [ ]:
gc=pd.read_excel(IO,sheet_name='Total_Supply',index_col=0)
dom=pd.read_excel(IO,sheet_name='Domestic_IO',index_col=0)
imp=pd.read_excel(IO,sheet_name='Import_IO',index_col=0)
labels=pd.read_excel(IO,sheet_name='Sector_List',index_col=0)
idx=list(range(64)); final=['Household','Government','Investment','Exports']
assert labels.index.tolist()==idx and all(i in gc.index for i in idx)
Z=gc.loc[idx,idx].to_numpy(float)
F=gc.loc[idx,final].to_numpy(float)
lab=gc.loc['Compensation of employees',idx].to_numpy(float)
cap=gc.loc['Gross operating surplus',idx].to_numpy(float)
product_tax=gc.loc['Net product taxes',idx].to_numpy(float)
other_tax=gc.loc['Other net production taxes',idx].to_numpy(float)
final_tax=gc.loc['Net product taxes',final].to_numpy(float)
imports=gc.loc['Imports / supply balancing',idx].to_numpy(float)
g=dom.loc['Domestic output',idx].to_numpy(float)
assert Z.shape==(64,64) and F.shape==(64,4)
assert all(np.isfinite(v).all() for v in [Z,F,lab,cap,product_tax,other_tax,final_tax,imports,g])
verify('Total intermediate transactions = domestic + imported',Z,dom.loc[idx,idx].to_numpy(float)+imp.loc[idx,idx].to_numpy(float))
verify('Total final use = domestic + imported',F,dom.loc[idx,final].to_numpy(float)+imp.loc[idx,final].to_numpy(float))
verify('Import vector = imported intermediate + final use',imports,imp.loc[idx,idx].sum(axis=1).to_numpy(float)+imp.loc[idx,final].sum(axis=1).to_numpy(float))
verify('Output = TurkStat P.1 total economy',g.sum(),float(tax_raw.iat[5,13]))
verify('Labour = TurkStat D.1 production uses',lab.sum(),float(tax_raw.iat[17,2]))
verify('Capital = TurkStat B.2g/B.3g',cap.sum(),float(tax_raw.iat[22,2]))
verify('Net product taxes = TurkStat D21-D31',product_tax.sum()+final_tax.sum(),float(tax_raw.iat[11,13]))
verify('Other net production taxes = TurkStat D29-D39',other_tax.sum(),float(tax_raw.iat[19,2])-float(tax_raw.iat[21,13]))
print('Source flows and 2023 revision consistency verified.')

## 4. Budget-consistent SAM construction

The construction follows Hosoe et al. (2010), with separate net product taxes and other net production taxes. Private, government and foreign saving close the corresponding accounts.

In [ ]:
def construct_sam(Z,F,lab,cap,production_tax,product_tax,tax_final,imports,TD,sector_codes,tariff=None):
    ac=sector_codes+['LAB','CAP','PTAX','CTAX','DTAX']+(['TRF'] if tariff is not None else [])+['HOH','GOV','INV','EXT']
    s=pd.DataFrame(0.0,index=ac,columns=ac)
    s.loc[sector_codes,sector_codes]=Z
    s.loc[sector_codes,['HOH','GOV','INV','EXT']]=F
    s.loc['LAB',sector_codes]=lab
    s.loc['CAP',sector_codes]=cap
    
    s.loc['PTAX',sector_codes]=production_tax
    s.loc['CTAX',sector_codes]=product_tax
    s.loc['CTAX',['HOH','GOV','INV','EXT']]=tax_final
    s.loc['EXT',sector_codes]=imports
    if tariff is not None:s.loc['TRF',sector_codes]=tariff
    s.loc['HOH','LAB']=np.sum(lab)
    s.loc['HOH','CAP']=np.sum(cap)
    for t in ['PTAX','CTAX']:
        s.loc['GOV',t]=s.loc[t].sum()
    if tariff is not None:s.loc['GOV','TRF']=np.sum(tariff)
    s.loc['DTAX','HOH']=TD
    s.loc['GOV','DTAX']=TD
    
    s.loc['INV','EXT']=s.loc['EXT'].sum()-s['EXT'].sum()
    s.loc['INV','HOH']=s.loc['HOH'].sum()-s['HOH'].sum()
    s.loc['INV','GOV']=s.loc['GOV'].sum()-s['GOV'].sum()
    return s

## 5. Replicate the textbook example

The three-industry Japanese example in Hosoe et al. (2010), Tables 4.4–4.5.C, verifies the accounting construction independently of the Turkish inputs.

In [ ]:
jZ=np.array([[1558469,8427693,1496991],[2462949,130321011,68646492],[2273437,63503715,160713811]],float)
jF=np.array([[3965927,0,967198,72018],[64927161,459179,39070695,46597315],[231268309,85247038,90250845,10817384]],float)
j=construct_sam(jZ,jF,np.array([1372650,59034654,234353029]),np.array([6167952,33816669,149889160]),np.array([534232,14436136,19877410]),np.zeros(3),np.zeros(4),np.array([1974129,39254377,9096735]),44207200,['AGR','MAN','SRV'],np.array([144478,3690547,911]))
verify('Hosoe Japan: all account balances (million yen)',j.sum(axis=1),j.sum(axis=0),atol=0)
verify('Hosoe Table 4.5.C: three saving accounts',j.loc['INV',['HOH','GOV','EXT']],[140265517,-2815303,-7161476],atol=0)
verify('Hosoe Table 4.5.C: industry totals',j.loc[['AGR','MAN','SRV']].sum(axis=1),[16488296,352484802,644074539],atol=0)
verify('Hosoe Table 4.5.C: factor incomes',j.loc['HOH',['CAP','LAB']],[189873781,294760333],atol=0)
display(j.loc[['HOH','GOV','INV'],['CAP','LAB','PTAX','TRF','DTAX','HOH','GOV','EXT']])

## 6. Construct and audit the Turkish SAM

All 73 row and column totals must balance. Intermediate uses, final uses and factors must match the input data, and investment must equal total saving. Signed source values are preserved.

In [ ]:
sam=construct_sam(Z,F,lab,cap,other_tax,product_tax,final_tax,imports,TD,codes)
assert sam.shape==(73,73) and sam.index.equals(sam.columns)
assert 'TRF' not in sam.index
row_total=sam.sum(axis=1);col_total=sam.sum(axis=0)
verify('Turkey SAM: 73 row-column balances',row_total,col_total)
verify('Turkey SAM: sector total = output + imports',row_total.loc[codes],g+imports)
verify('Investment = private + government + foreign saving',sam['INV'].sum(),sam.loc['INV',['HOH','GOV','EXT']].sum())
verify('SAM intermediate transactions match the source',sam.loc[codes,codes],Z,atol=0)
verify('SAM final transactions match the source',sam.loc[codes,['HOH','GOV','INV','EXT']],F,atol=0)
verify('SAM factors match the source',sam.loc[['LAB','CAP'],codes],np.vstack([lab,cap]),atol=0)
balance=pd.DataFrame({'Row total':row_total,'Column total':col_total,'Difference':row_total-col_total})
balance['Absolute difference']=abs(balance['Difference'])
balance['Relative difference']=balance['Absolute difference']/np.maximum(np.maximum(abs(row_total),abs(col_total)),1)
balance['Within tolerance']=balance['Absolute difference']<=1e-3
balance.index.name='Account'
macro_groups={'SECTORS':codes,**{a:[a] for a in ['LAB','CAP','PTAX','CTAX','DTAX','HOH','GOV','INV','EXT']}}
macro=pd.DataFrame({c:{r:float(sam.loc[rs,cs].to_numpy().sum()) for r,rs in macro_groups.items()} for c,cs in macro_groups.items()})
negative=sam.stack().loc[lambda x:x<0].rename('Value (thousand TRY)').reset_index()
negative.columns=['Recipient account','Payer account','Value (thousand TRY)']
display(sam.loc['INV',['HOH','GOV','EXT']].to_frame('Saving (thousand TRY)'))
print('Maximum absolute balance gap (thousand TRY):',balance['Absolute difference'].max())
print('Negative cell count:',len(negative),'(retained without balancing adjustments)')
verify('PTAX preserves source other production taxes',sam.loc['PTAX',codes],other_tax,atol=0)
verify('CTAX preserves source product taxes',sam.loc['CTAX',codes+['HOH','GOV','INV','EXT']],np.r_[product_tax,final_tax],atol=0)
verify('Sum of the two tax accounts matches the combined source total',sam.loc['GOV',['PTAX','CTAX']].sum(),other_tax.sum()+product_tax.sum()+final_tax.sum())
verify('Final uses have zero PTAX',sam.loc['PTAX',['HOH','GOV','INV','EXT']],np.zeros(4),atol=0)


## 7. Accounts, tax bridge and transaction provenance

S00–S63 retain the source industry order. The account dictionary, tax bridge and transaction-level provenance document each aggregate flow.

In [ ]:
account_rows=[]
for i,c in enumerate(codes):
    account_rows.append([c,i,str(labels.iloc[i,1]),str(labels.iloc[i,2]),'Sector','IO Total_Supply: rows/columns 0–63'])
extra=[('LAB','Labour','Compensation of employees'),('CAP','Capital','Gross operating surplus / mixed income, including depreciation'),('PTAX','Other net production taxes','Production taxes less subsidies; D29-D39'),('CTAX','Net product taxes','Commodity taxes less subsidies; D21-D31; import duties are not separately identified'),('DTAX','Direct tax','TurkStat D.5 general government resources; HOH → DTAX → GOV'),('HOH','Representative household','All labour and capital income; representative private-sector scope'),('GOV','Government','Tax revenue, government consumption and residual government saving'),('INV','Investment-tasarruf','Investment and inventories; private, government and foreign saving'),('EXT','Rest of the world','IO imports/exports; excludes foreign factor income and transfers')]
for c,name,desc in extra:account_rows.append([c,None,'',name,'Factor' if c in ['LAB','CAP'] else 'Tax' if c in ['PTAX','CTAX','DTAX'] else 'Institution',desc])
accounts=pd.DataFrame(account_rows,columns=['Account','Previous code','NACE','Name','Group','Scope'])
bridge=[]
for i,c in enumerate(codes):bridge.append([c,str(labels.iloc[i,2]),product_tax[i],other_tax[i],product_tax[i]+other_tax[i],'I_industry_io_tables_2023.xlsx / Total_Supply'])
for i,c in enumerate(['HOH','GOV','INV','EXT']):bridge.append([c,final[i],final_tax[i],0.0,final_tax[i],'I_industry_io_tables_2023.xlsx / Total_Supply'])
transactions=[]
for recipient in sam.index:
    for payer in sam.columns:
        v=float(sam.loc[recipient,payer])
        if v==0:continue
        if recipient=='INV':method='Row-column residual (closure)';source='Hosoe 4.2–4.3'
        elif recipient=='DTAX' or (recipient=='GOV' and payer=='DTAX'):method='External data / tax transfer';source='TurkStat t_2023!L48'
        elif recipient=='HOH' or (recipient=='GOV' and payer in ['PTAX','CTAX']):method='Total income transfer (assumption)';source='Hosoe 4.2–4.3 and IO'
        elif recipient=='PTAX':method='Separate transfer of other net production taxes';source='IO Total_Supply: Other net production taxes (D29-D39)'
        elif recipient=='CTAX':method='Separate transfer of net product taxes';source='IO Total_Supply: Net product taxes (D21-D31)'
        else:method='Direct transfer of IO flow';source='IO Total_Supply'
        transactions.append([recipient,payer,v,method,source])
transactions=pd.DataFrame(transactions,columns=['Recipient','Payer','Value_thousand_TRY','Construction_method','Source'])
sam.index.name='Recipient_Payer'
sam.to_csv(OUT/'SAM_2023_64_sectors.csv',encoding='utf-8-sig',float_format='%.12f')
transactions.to_csv(OUT/'SAM_2023_transaction_sources.csv',index=False,encoding='utf-8-sig',float_format='%.12f')

## 8. Workbook construction

The full matrix, macro aggregation, account dictionary, tax bridge and balance checks are exported. Formula caches are calculated directly from the validated Python arrays.

In [ ]:
def xlcol(n):
    out=''
    while n:n,r=divmod(n-1,26);out=chr(65+r)+out
    return out
ac=list(sam.index);positions={a:i for i,a in enumerate(ac)};N=len(ac)
def row(a):return positions[a]+6
def col(a):return xlcol(positions[a]+2)
def cell(r,c):return f'{col(c)}{row(r)}'
lastcol=xlcol(N+1);totalcol=xlcol(N+2);lastrow=N+5;totalrow=N+6
main_name=f'SAM_{N}'
def spec(name,title,subtitle,headers,rows,numeric=False):
    width=len(headers)
    values=[[None]*width for _ in range(4)]+[headers]+rows
    values[1][0]=title;values[2][0]=subtitle
    return {'name':name,'values':values,'numeric':numeric,'formulas':[],'formats':[]}
def form(s,where,formula,value=None):
    s['formulas'].append({'cell':where,'formula':formula})
    if value is not None:expected.append({'sheet':s['name'],'cell':where,'value':float(value)})
expected=[]
main=spec(main_name,'Turkey 2023 social accounting matrix','Thousand TRY; rows receive and columns share. 64 sectors + 9 accounts; separate PTAX and CTAX.',['Account']+ac+['TOTAL'],[[r]+sam.loc[r].tolist()+[None] for r in ac]+[['TOTAL']+[None]*(N+1)],True)
for a in ac:
    form(main,f'{totalcol}{row(a)}',f'=SUM(B{row(a)}:{lastcol}{row(a)})',row_total[a])
    form(main,f'{col(a)}{totalrow}',f'=SUM({col(a)}6:{col(a)}{lastrow})',col_total[a])
form(main,f'{totalcol}{totalrow}',f'=SUM({totalcol}6:{totalcol}{lastrow})',sam.to_numpy().sum())
for i,c in enumerate(codes):
    form(main,cell('PTAX',c),f"='Tax_Bridge'!D{i+6}",sam.loc['PTAX',c])
    form(main,cell('CTAX',c),f"='Tax_Bridge'!C{i+6}",sam.loc['CTAX',c])
for i,c in enumerate(['HOH','GOV','INV','EXT']):
    form(main,cell('CTAX',c),f"='Tax_Bridge'!C{i+70}",sam.loc['CTAX',c])
for factor in ['LAB','CAP']:
    form(main,cell('HOH',factor),f'=SUM({cell(factor,codes[0])}:{cell(factor,codes[-1])})',sam.loc['HOH',factor])
form(main,cell('DTAX','HOH'),"='Tax_Bridge'!B78",TD)
form(main,cell('GOV','DTAX'),f'={cell("DTAX","HOH")}',TD)
for taxacc in ['PTAX','CTAX']:
    form(main,cell('GOV',taxacc),f'=SUM(B{row(taxacc)}:{lastcol}{row(taxacc)})',sam.loc['GOV',taxacc])
form(main,cell('INV','EXT'),f'=SUM({cell("EXT",codes[0])}:{cell("EXT",codes[-1])})-SUM({cell(codes[0],"EXT")}:{cell(codes[-1],"EXT")})-{cell("CTAX","EXT")}',sam.loc['INV','EXT'])
form(main,cell('INV','HOH'),f'={cell("HOH","LAB")}+{cell("HOH","CAP")}-SUM({cell(codes[0],"HOH")}:{cell(codes[-1],"HOH")})-{cell("CTAX","HOH")}-{cell("DTAX","HOH")}',sam.loc['INV','HOH'])
form(main,cell('INV','GOV'),f'={cell("GOV","PTAX")}+{cell("GOV","CTAX")}+{cell("GOV","DTAX")}-SUM({cell(codes[0],"GOV")}:{cell(codes[-1],"GOV")})-{cell("CTAX","GOV")}',sam.loc['INV','GOV'])
main['previews']=['A1:F12',f'BN70:{totalcol}{totalrow}']
main['formats']=[{'range':f'A70:{totalcol}{lastrow}','format':{'fill':'#F0F4F8'}},{'range':f'A{totalrow}:{totalcol}{totalrow}','format':{'font':{'bold':True},'fill':'#DCE6EE'}},{'range':f'{totalcol}5:{totalcol}{totalrow}','format':{'font':{'bold':True}}}]
macro_names=list(macro_groups);M=len(macro_names);mtcol=xlcol(M+2);mtrow=M+6;mlcol=xlcol(M+1)
mac=spec('Macro_SAM','Macro SAM view','Thousand TRY; SECTORS aggregates 64 sectors for this display only.',['Account']+macro_names+['TOTAL'],[[r]+[None]*(M+1) for r in macro_names]+[['TOTAL']+[None]*(M+1)],True)
for ri,r in enumerate(macro_names):
    for ci,c in enumerate(macro_names):
        rs,cs=macro_groups[r],macro_groups[c]
        rng=f"'{main_name}'!{cell(rs[0],cs[0])}:{cell(rs[-1],cs[-1])}"
        form(mac,f'{xlcol(ci+2)}{ri+6}',f'=SUM({rng})',macro.loc[r,c])
    form(mac,f'{mtcol}{ri+6}',f'=SUM(B{ri+6}:{mlcol}{ri+6})',macro.loc[r].sum())
for ci,c in enumerate(macro_names):
    cc=xlcol(ci+2);form(mac,f'{cc}{mtrow}',f'=SUM({cc}6:{cc}{mtrow-1})',macro[c].sum())
form(mac,f'{mtcol}{mtrow}',f'=SUM(B{mtrow}:{mlcol}{mtrow})',macro.to_numpy().sum())
mac['previews']=[f'A1:F{mtrow}',f'G5:{mtcol}{mtrow}']
mac['formats']=[{'range':f'A{mtrow}:{mtcol}{mtrow}','format':{'font':{'bold':True},'fill':'#DCE6EE'}}]
acc=spec('Accounts','Account dictionary','S00–S63 correspond to source sector codes 0–63. Other identifiers name institutional accounts.',accounts.columns.tolist(),accounts.astype(object).where(pd.notna(accounts),None).values.tolist())
acc['widths']={'0':13,'1':14,'2':15,'3':70,'4':14,'5':90}
acc['formats']=[{'range':f'D6:F{lastrow}','format':{'wrapText':True}},{'range':f'A6:F{lastrow}','format':{'rowHeight':44}}]
acc['previews']=['A1:F10',f'A69:F{lastrow}']
taxsheet=spec('Tax_Bridge','Transfer of tax data to separate SAM accounts','Thousand TRY. CTAX: net product taxes. PTAX: other net production taxes. Import duties are not separate.',['Account','Name','CTAX: net product taxes','PTAX: other net production taxes','PTAX + CTAX','Source'],bridge,True)
while len(taxsheet['values'])<86:taxsheet['values'].append([None]*6)
taxsheet['values'][76]=['Transaction','Value','Unit','Institutional side','Source cell','Source workbook / sheet']
for rr,code,source_row,source_col,desc in [(77,'D.5',r5,11,'General government resources'),(78,'D.5 HOH',r5,3,'Household uses; excluded from the model input'),(79,'D.51',r51,11,'General government resources'),(80,'D.59',r59,11,'General government resources')]:
    taxsheet['values'][rr]=[code,float(tax_raw.iat[source_row,source_col]),'thousand TRY',desc,f'{chr(65+source_col)}{source_row+1}','TurkStat 2023 institutional accounts / t_2023']
taxsheet['values'][83]=['TurkStat statistical release','https://veriportali.tuik.gov.tr/tr/press/57929',None,None,None,None]
taxsheet['values'][84]=['Release / reference period','9 October 2025 / 2023',None,None,None,None]
taxsheet['values'][85]=['Method','Hosoe et al. (2010), Chapter 4, pp. 47–56; Japan Tables 4.4–4.5.C',None,None,None,None]
for i,b in enumerate(bridge):form(taxsheet,f'E{i+6}',f'=C{i+6}+D{i+6}',b[4])
taxsheet['widths']={'0':17,'1':62,'2':23,'3':30,'4':23,'5':64}
taxsheet['formats']=[{'range':'A6:F73','format':{'rowHeight':42}},{'range':'B6:B73','format':{'wrapText':True}},{'range':'A77:F81','format':{'wrapText':True,'rowHeight':42}}]
taxsheet['previews']=['A1:F9','A69:F81']
audit=spec('Checks','Account balance checks','Thousand TRY; absolute tolerance 0.001 thousand TRY = 1 TRY. Source flows preserved.',['Account','Row total','Column total','Difference','Absolute difference','Relative difference','Result'],[[a,None,None,None,None,None,None] for a in ac],True)
for a in ac:
    rr=row(a)
    form(audit,f'B{rr}',f"='{main_name}'!{totalcol}{rr}",row_total[a])
    form(audit,f'C{rr}',f"='{main_name}'!{col(a)}{totalrow}",col_total[a])
    form(audit,f'D{rr}',f'=B{rr}-C{rr}',balance.loc[a,'Difference'])
    form(audit,f'E{rr}',f'=ABS(D{rr})',balance.loc[a,'Absolute difference'])
    form(audit,f'F{rr}',f'=E{rr}/MAX(ABS(B{rr}),ABS(C{rr}),1)',balance.loc[a,'Relative difference'])
    form(audit,f'G{rr}',f'=IF(E{rr}<=0.001,"Pass","Difference detected")')
audit['formats']=[{'range':f'D6:F{lastrow}','numberFormat':'0.00E+00'}]
audit['previews']=['A1:G10',f'A70:G{lastrow}']
job={'output':str(OUT/'SAM_2023_64_sectors.xlsx'),'main_sheet':main_name,'sensitivity_cells':[cell('INV','HOH'),cell('INV','GOV'),f'{col("INV")}{totalrow}'],'sheets':[main,mac,acc,taxsheet,audit],'expected':expected}

## 9. Export and round-trip checks

The saved workbook is reopened, checked for spreadsheet errors and compared with the Python SAM. The input files are rechecked after export.

In [ ]:
import sys
for parent in [Path.cwd(),*Path.cwd().parents]:
    if (parent/'tools/public_excel_export.py').is_file():
        sys.path.insert(0,str(parent/'tools'));break
from public_excel_export import export_jobs
export_jobs([job])
book=pd.ExcelFile(OUT/'SAM_2023_64_sectors.xlsx')
assert book.sheet_names==['SAM_73','Macro_SAM','Accounts','Tax_Bridge','Checks']
saved=pd.read_excel(book,sheet_name='SAM_73',header=4,index_col=0)
assert saved.index[:73].tolist()==ac and saved.columns[:73].tolist()==ac
verify('Saved Excel = Python SAM',saved.loc[ac,ac].to_numpy(float),sam.to_numpy())
verify('Saved Excel: 73 account balances',saved.loc[ac,ac].sum(axis=1),saved.loc[ac,ac].sum(axis=0))
for sh in book.sheet_names:
    frame=pd.read_excel(book,sheet_name=sh,header=None)
    errors={'#REF!','#DIV/0!','#VALUE!','#NAME?','#N/A','#NUM!','#NULL!','#SPILL!','#CALC!'}
    assert not any(isinstance(v,str) and v in errors for v in frame.to_numpy().ravel()),sh
for e in manifest:
    if e.get('sha256'): assert hashlib.sha256((ROOT/e['file']).read_bytes()).hexdigest()==e['sha256']
run={'stage':2,'year':2023,'unit':'thousand TRY','accounts':73,'sectors':64,'direct_tax_D5':TD,'net_production_tax_PTAX':float(sam.loc['GOV','PTAX']),'net_product_tax_CTAX':float(sam.loc['GOV','CTAX']),'net_production_product_tax':float(sam.loc['GOV',['PTAX','CTAX']].sum()),'savings':{k:float(sam.loc['INV',k]) for k in ['HOH','GOV','EXT']},'investment':float(sam['INV'].sum()),'maximum_balance_error_bin_TL':float(balance['Absolute difference'].max()),'negative_cells':len(negative),'balancing_adjustment':'None; source flows preserved','checks':checks,'python_numpy':np.__version__,'python_pandas':pd.__version__,'output_sha256':hashlib.sha256((OUT/'SAM_2023_64_sectors.xlsx').read_bytes()).hexdigest()}
(OUT/'execution_log.json').write_text(json.dumps(run,ensure_ascii=False,indent=2),encoding='utf-8')
print('SAM and all checks completed.')

## Interpretation and next stage

The representative private account receives aggregate labour and capital income. D.5 covers general-government current taxes on income and wealth. The next stage allocates this accounting scope across five household groups using restricted HBS distributional shares.

## Composite-goods representation

The model uses the composite-goods SAM produced above. An alternative analytical presentation is unnecessary for this replication pipeline.

In [ ]:
# The model uses the original composite-goods SAM above; the alternative analytical presentation is omitted from this replication package.
